In [ ]:
import pandas as pd
import numpy as np
df_vix = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/Data/PKL/vix-1m.pkl")

In [ ]:
df_vix.rename(columns={"Close":"VIX"},inplace=True)
# df_vix.index = pd.to_datetime(df_vix.index)
# df_vix["Date"]= pd.to_datetime(df_vix.index)
df_vix = df_vix.loc[df_vix["Date"]>"2020-05-31"]

df_vix["VIX(t-1)"] = df_vix["VIX"].shift(periods=1)
df_vix["VIX(t-1)"].ffill(inplace=True)

df_vix["VIX return"]=np.log(df_vix["VIX"]/df_vix["VIX(t-1)"])
df_vix["VIX return"].ffill(inplace=True)


df_vix["VIX NR"] = df_vix["VIX return"].mean()
df_vix["VIX AR"] = df_vix["VIX return"] - df_vix["VIX NR"]


In [ ]:
import statistics
vix_stdev = statistics.stdev(df_vix["VIX AR"].dropna())
vix_mean = df_vix["VIX AR"].mean()
len(df_vix.loc[(df_vix["VIX AR"] > vix_mean+vix_stdev) | (df_vix["VIX AR"] < vix_mean-vix_stdev)])
vix = df_vix.loc[(df_vix["VIX AR"] > vix_mean+vix_stdev) | (df_vix["VIX AR"] < vix_mean-vix_stdev)]
df_vix['VIX AR']

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(font_scale=1.5)

ax = sns.distplot(df_vix['VIX AR'], hist=True,color='#388E3C')
kde_x, kde_y = ax.lines[0].get_data()

x0 = vix_mean-vix_stdev
x1 =vix_mean+vix_stdev
#plotting the two lines
p1 = plt.axvline(x=x0,color='#EF9A9A')
p2 = plt.axvline(x=x1,color='#EF9A9A')
plt.title("Minute Frequency\nOne Standard diviations Above or below the mean AR\nPress Release")

ax.fill_between(kde_x, kde_y, where=(kde_x<x0) | (kde_x>x1) , 
                interpolate=True, color='#EF9A9A')